## SETUPS ##

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


## SCRAPPING ##

In [2]:
ABBYSINIA_APP_ID = 'com.boa.boaMobileBanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    ABBYSINIA_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Abbysinia App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

Abbysinia App Info
App Title   : BoA Mobile
Current Score: 4.4006343
Total Ratings: 9,301
Total Reviews: 1,470
Installs     : 1,000,000+


In [28]:
# Step 2: Scrape reviews
print(f"Scraping reviews for Abbysinia...")

result, continuation_token = reviews(
    ABBYSINIA_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count= 500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Abbysinia...
Collected 500 raw reviews


In [29]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: ab5c2323-7ea1-45f7-8d2b-53479e0b57b5
  userName: faju show
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjXjWga4UYcChWMgwsFfV5aMlffQhnFBC5WsCMmdXVfgjI4MAuoj
  content: It gave me a headache. bad bad bad
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: None
  at: 2026-05-18 17:21:12
  replyContent: None
  repliedAt: None
  appVersion: None


In [30]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Abbysinia Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,ab5c2323-7ea1-45f7-8d2b-53479e0b57b5,It gave me a headache. bad bad bad,1,2026-05-18 17:21:12,Abbysinia Bank,Google Play
1,325e6111-ef98-4b7d-a8ce-156557d9c50f,bankn abstract fayida,5,2026-05-18 16:03:28,Abbysinia Bank,Google Play
2,e1dfd8e3-02ad-4d83-80e8-0fa5ec2c06e3,Tilku,5,2026-05-18 03:16:59,Abbysinia Bank,Google Play
3,ea419fbc-41fc-4211-b3b4-84e87d2952a4,the transaction is not working???? fix it,1,2026-05-18 01:13:03,Abbysinia Bank,Google Play
4,0fe83567-9471-413c-b722-a24757bb5d82,sometimes The App Is not goibg through,3,2026-05-17 15:21:31,Abbysinia Bank,Google Play


In [31]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [32]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  280  ████████████████████████████████████████████████████████
  4 stars:   37  ███████
  3 stars:   19  ███
  2 stars:   16  ███
  1 stars:  148  █████████████████████████████


In [33]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-18 17:21:12
1   2026-05-18 16:03:28
2   2026-05-18 03:16:59
3   2026-05-18 01:13:03
4   2026-05-17 15:21:31
5   2026-05-16 12:35:22
6   2026-05-16 00:10:06
7   2026-05-15 21:07:21
8   2026-05-15 15:01:09
9   2026-05-14 21:18:44

Date dtype: datetime64[us]


## DATA QUALITY AUDIT ##


In [34]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [35]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 90
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [36]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-18 17:21:12
  Target format: YYYY-MM-DD (string or date object)


In [37]:
df_raw = pd.DataFrame(raw_data)
df = df_raw.copy()
print(f"Starting with: {len(df)} reviews")

Starting with: 500 reviews


In [38]:
before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


In [39]:
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Removed 0 duplicate reviews
Remaining: 500 reviews


In [40]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-18 17:21:12
1   2026-05-18 16:03:28
2   2026-05-18 03:16:59
dtype: datetime64[us]

After normalization:
0    2026-05-18
1    2026-05-18
2    2026-05-18
dtype: str

Date range: 2025-03-02 to 2026-05-18


In [41]:
import re
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


In [42]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


In [43]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,It gave me a headache. bad bad bad,1,2026-05-18,Abbysinia Bank,Google Play
1,bankn abstract fayida,5,2026-05-18,Abbysinia Bank,Google Play
2,Tilku,5,2026-05-18,Abbysinia Bank,Google Play
3,the transaction is not working???? fix it,1,2026-05-18,Abbysinia Bank,Google Play
4,sometimes The App Is not goibg through,3,2026-05-17,Abbysinia Bank,Google Play
5,"The worst app, also bank am begging for my own...",1,2026-05-16,Abbysinia Bank,Google Play
6,was Good 🙏,5,2026-05-16,Abbysinia Bank,Google Play
7,cool,5,2026-05-15,Abbysinia Bank,Google Play
8,Its Good,5,2026-05-15,Abbysinia Bank,Google Play
9,good,5,2026-05-14,Abbysinia Bank,Google Play


In [45]:
# Save to CSV
import os
os.makedirs('data/processed', exist_ok=True)

output_path = 'data/processed/abbysinia_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: data/processed/abbysinia_bank_reviews_clean.csv


In [46]:
import pandas as pd

# 1. Convert the list of scraped reviews into a DataFrame
df = pd.DataFrame(result)

# 2. Save the DataFrame to a CSV file
df.to_csv('abbysinia_bank_reviews.csv', index=False, encoding='utf-8')

print("CSV file created successfully!")


CSV file created successfully!
